In [1]:
import os
from skyfield.api import load, wgs84
from datetime import timedelta

# Load timescales and orbital/planetary data
ts = load.timescale()
t0 = ts.now()
t1 = ts.utc(t0.utc_datetime() + timedelta(hours=24))

# Load JPL ephemeris for solar illumination math (Nighttime optical filtering)
eph = load('de421.bsp')
sun = eph['sun']
earth = eph['earth']

file_path = 'active_sats.txt'

if not os.path.exists(file_path):
    print(f"ERROR: '{file_path}' not found in the local directory.")
else:
    satellites = load.tle_file(file_path)
    
    if len(satellites) == 0:
        print(f"ERROR: '{file_path}' loaded 0 satellites.")
    else:
        # Calculate the age of the orbital data to warn of positional drift
        sample_sat = satellites[0]
        data_age_days = t0 - sample_sat.epoch
        
        print("=========================================================================")
        print("                     OVERHEAD COLLECTION FORECAST                        ")
        print("=========================================================================")
        if abs(data_age_days) > 3.0:
            print(f"[WARNING] TLE Data is {abs(data_age_days):.1f} days old. Atmospheric drag drift expected.")
            print("[ACTION]  Download fresh active_sats.txt before field deployment.")
        else:
            print(f"[OK] TLE Data is {abs(data_age_days):.1f} days old. Prediction confidence high.")
        print("=========================================================================\n")

        # Define constellations
        optical_constellations = ['GAOFEN', 'JILIN']
        sar_identifiers = [
            'YAOGAN-29', 'YAOGAN-33', 
            'YAOGAN-1 ', 'YAOGAN-3 ', 'YAOGAN-10', 'YAOGAN-13', 'YAOGAN-18'
        ]
        
        optical_sats = [sat for sat in satellites if any(name in sat.name for name in optical_constellations)]
        sar_sats = [sat for sat in satellites if any(name in sat.name for name in sar_identifiers)]
        
        # Coordiantes input
        ground_station = wgs84.latlon(33.1893, -87.5652)
        observer = earth + ground_station # Combines Earth center with surface topo for sun math
        
        overhead_intervals = []
        
        # Daylight filterting
        for sat in optical_sats:
            times, events = sat.find_events(ground_station, t0, t1, altitude_degrees=45.0)
            rise_time = None
            for ti, event in zip(times, events):
                if event == 0:  
                    rise_time = ti.tt
                elif event == 2:  
                    if rise_time is not None:
                        # Calculate midpoint of the satellite pass
                        mid_tt = (rise_time + ti.tt) / 2.0
                        mid_time = ts.tt(jd=mid_tt)
                        
                        # Calculate the Sun's altitude from coordiantes at that exact moment
                        astrometric = observer.at(mid_time).observe(sun)
                        alt, az, distance = astrometric.apparent().altaz()
                        
                        # Only add to threat list if the sun is above -6 degrees (Twilight/Daylight)
                        if alt.degrees > -6.0:
                            overhead_intervals.append([rise_time, ti.tt])
                        rise_time = None
                        
        # SAR Processing (ignores daylight filtering)
        for sat in sar_sats:
            times, events = sat.find_events(ground_station, t0, t1, altitude_degrees=20.0)
            rise_time = None
            for ti, event in zip(times, events):
                if event == 0:  
                    rise_time = ti.tt
                elif event == 2:  
                    if rise_time is not None:
                        overhead_intervals.append([rise_time, ti.tt])
                        rise_time = None
                        
        # Sort and merge overlapping windows
        overhead_intervals.sort(key=lambda x: x[0])
        merged_threats = []
        for interval in overhead_intervals:
            if not merged_threats or merged_threats[-1][1] < interval[0]:
                merged_threats.append(interval)
            else:
                merged_threats[-1][1] = max(merged_threats[-1][1], interval[1])
                
        # Brief Output
        timeline = []
        current_time = t0.tt
        end_time = t1.tt
        
        for threat_start, threat_end in merged_threats:
            # If there is a gap before the next threat, log it as a CLEAR window
            if current_time < threat_start:
                timeline.append((current_time, threat_start, "CLEAR"))
            # Log the THREAT window
            timeline.append((threat_start, threat_end, "THREAT"))
            current_time = max(current_time, threat_end)
            
        # Add the final clear window if the forecast ends without a threat
        if current_time < end_time:
            timeline.append((current_time, end_time, "CLEAR"))
            
       # Schedule output
        print("TIME (UTC)                   | STATUS              | DURATION")
        print("-" * 68)
        
        for start_jd, end_jd, status in timeline:
            start_obj = ts.tt(jd=start_jd)
            end_obj = ts.tt(jd=end_jd)
            duration_minutes = (end_jd - start_jd) * 24 * 60
            
            # Format date and time
            start_str = start_obj.utc_strftime('%m-%d %H:%M')
            end_str = end_obj.utc_strftime('%m-%d %H:%M')
            
            if status == "THREAT":
                print(f"{start_str} to {end_str} | [X] SAT OVERHEAD  | {duration_minutes:5.1f} min")
            elif status == "CLEAR":
                if duration_minutes < 15.0:
                    print(f"{start_str} to {end_str} | [-] MICRO-GAP     | {duration_minutes:5.1f} min (Too short)")
                else:
                    print(f"{start_str} to {end_str} | [O] ALL CLEAR     | {duration_minutes:5.1f} min")

                     OVERHEAD COLLECTION FORECAST                        
[WARNING] TLE Data is 3.6 days old. Atmospheric drag drift expected.
[ACTION]  Download fresh active_sats.txt before field deployment.

TIME (UTC)                   | STATUS              | DURATION
--------------------------------------------------------------------
09-28 20:10 to 09-28 20:11 | [-] MICRO-GAP     |   0.4 min (Too short)
09-28 20:11 to 09-28 20:16 | [X] SAT OVERHEAD  |   5.3 min
09-28 20:16 to 09-28 20:25 | [-] MICRO-GAP     |   8.6 min (Too short)
09-28 20:25 to 09-28 20:26 | [X] SAT OVERHEAD  |   1.8 min
09-28 20:26 to 09-28 21:03 | [O] ALL CLEAR     |  36.8 min
09-28 21:03 to 09-28 21:05 | [X] SAT OVERHEAD  |   1.3 min
09-28 21:05 to 09-28 21:09 | [-] MICRO-GAP     |   4.3 min (Too short)
09-28 21:09 to 09-28 21:10 | [X] SAT OVERHEAD  |   1.2 min
09-28 21:10 to 09-28 22:04 | [O] ALL CLEAR     |  54.5 min
09-28 22:04 to 09-28 22:06 | [X] SAT OVERHEAD  |   1.6 min
09-28 22:06 to 09-28 23:10 | [O] 